In [127]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client
from tqdm.auto import tqdm

In [42]:
#01 Menentukan direktori data
HERE = Path.cwd()
# Menyiapkan path ke folder data
DATA = HERE / "data"

In [52]:
#02 Memuat data dari NPASS
NPASSnpSpecies = pd.read_csv(DATA / "00NPASSnp-species.csv", sep="\t")
NPASSnp = pd.read_csv(DATA / "00NPASSnp.csv", sep="\t")
NPASSspecies = pd.read_csv(DATA / "00NPASSspecies.csv", sep="\t")

In [ ]:
#03 Inspeksi bentuk dari data
print(NPASSnpSpecies.shape)
NPASSnpSpecies.head()

In [ ]:
#04 Inspeksi bentuk dari data
print(NPASSnp.shape)
NPASSnp.head()

In [ ]:
#05 Inspeksi bentuk dari data
print(NPASSspecies.shape)
NPASSspecies.head()

In [60]:
#06 Pilih Kolom yang relevan
NPASSnpSpecies = NPASSnpSpecies[['org_id', 'np_id']]
NPASSnp = NPASSnp[['np_id', 'pref_name', 'chembl_id', 'pubchem_cid']]
NPASSspecies = NPASSspecies[['org_id', 'org_name']]

In [ ]:
#07 Merge natural product name information
NPASSmerged1 = pd.merge(NPASSnpSpecies, NPASSnp, on="np_id")
print(NPASSmerged1.shape)
NPASSmerged1.head()

In [ ]:
#08 Merge species name
NPASSmerged2 = pd.merge(NPASSmerged1, NPASSspecies, on="org_id")
print(NPASSmerged2.shape)
NPASSmerged2.head()

In [ ]:
#09 Seleksi data dengan informasi chembl_id saja
NPASSmerged2 = NPASSmerged2[NPASSmerged2['chembl_id'] != "n.a."]
print(NPASSmerged2.shape)
NPASSmerged2.head()

In [ ]:
#10 Baca data dari GBIF
GBIFidnPlant = pd.read_csv(DATA / "00GBIFindoSpecies.csv", sep="\t")
print(GBIFidnPlant.shape)
GBIFidnPlant.tail()

In [ ]:
#11 Seleksi species NPASS dengan GBIF 
idnPlantNP = NPASSmerged2[NPASSmerged2['org_name'].isin(GBIFidnPlant['species'])]
idnPlantNP.reset_index(drop=True, inplace=True)
print(idnPlantNP.shape)
idnPlantNP.head()

In [ ]:
#12 Group by np_id and create species list while keeping all columns
idnPlantNP_grouped = idnPlantNP.groupby('pref_name').agg({
    'org_name': lambda x: list(set(x)),  # Create species list
    'chembl_id': 'first',  # Keep the first occurrence of chembl_id
    'pubchem_cid': 'first',  # Keep the first occurrence of pubchem_cid
}).reset_index()

idnPlantNP_grouped = idnPlantNP_grouped.rename(columns={'chembl_id': 'molecule_chembl_id'})
print(idnPlantNP_grouped.shape)
idnPlantNP_grouped.head()

In [ ]:
#13 Memilih 200 sampel acak dari idnPlantNP_grouped untuk menghemat waktu dalam workshop
idnPlantNP_grouped_sample = idnPlantNP_grouped.sample(n=200, random_state=42)
print(idnPlantNP_grouped_sample.shape)
idnPlantNP_grouped_sample.head()

In [146]:
#14 Menyiapkan pengambilan data struktur molekul dari ChEMBL berdasarkan ID yang ada di dataset
compounds_api = new_client.molecule
compounds_provider = compounds_api.filter(
    molecule_chembl_id__in=list(idnPlantNP_grouped_sample["molecule_chembl_id"])
).only("molecule_chembl_id", "molecule_structures")

In [147]:
#15 Eksekusi pengambilan data
compounds = list(tqdm(compounds_provider))

  0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
#16 Membuat DataFrame dari data struktur molekul dan menampilkan informasi awal
compounds_df = pd.DataFrame.from_records(
    compounds,
)
print(f"DataFrame shape: {compounds_df.shape}")
compounds_df.head()

In [149]:
#17 Menghapus data yang kosong, apabila ada
compounds_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

DataFrame shape: (200, 2)


In [150]:
#18 Identifikasi struktur data dari ChEMBL
compounds_df.iloc[0].molecule_structures.keys()

dict_keys(['canonical_smiles', 'molfile', 'standard_inchi', 'standard_inchi_key'])

In [151]:
#19 Isolasi canonical smiles dari data compound ChEMBL
canonical_smiles = []

for i, compounds in compounds_df.iterrows():
    try:
        canonical_smiles.append(compounds["molecule_structures"]["canonical_smiles"])
    except KeyError:
        canonical_smiles.append(None)

compounds_df["smiles"] = canonical_smiles
compounds_df.drop("molecule_structures", axis=1, inplace=True)
print(f"DataFrame shape: {compounds_df.shape}")

DataFrame shape: (200, 2)


In [ ]:
#20 Merge DataFrames
output_df = pd.merge(idnPlantNP_grouped_sample, compounds_df, on="molecule_chembl_id")

# Reset row indices
output_df.reset_index(drop=True, inplace=True)
print(f"Dataset with {output_df.shape[0]} entries.")
output_df.head(10)

In [156]:
#21 save output data dalam file
output_df.to_csv(DATA / "01sampledIndonesiaPlantNP.csv")